In [16]:
!pip install google-adk google-cloud-aiplatform google-cloud-storage --quiet

In [17]:
import hashlib
import vertexai
from google.cloud import storage
from google.api_core import exceptions

# Project config
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

# Create a unique staging bucket name from the project ID
id_hash = hashlib.sha256(PROJECT_ID.encode()).hexdigest()[:4]
bucket_name = f"agent-staging-{id_hash}"

print(f"Target Bucket Name: {bucket_name}")

# Create bucket if it doesn't exist
storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.get_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' already exists. Skipping creation.")
except exceptions.NotFound:
    print(f"Bucket '{bucket_name}' not found. Creating...")
    bucket = storage_client.create_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' created successfully.")
except Exception as e:
    print(f"An error occurred: {e}")

STAGING_BUCKET = f"gs://{bucket_name}"

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

Target Bucket Name: agent-staging-c19a
Bucket 'agent-staging-c19a' already exists. Skipping creation.
Project ID: qwiklabs-gcp-00-11b9d8ae6979
Location: us-central1
Staging Bucket: gs://agent-staging-c19a


In [18]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

In [19]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

from google.adk.agents import Agent

general_agent = Agent(
    name="helpful_assistant",
    model="gemini-2.5-flash",
    description="A friendly, helpful assistant that can answer general questions and tell jokes.",
    instruction="""
    You are a friendly and helpful assistant. Answer user questions clearly
    and concisely. If you don't know the answer to something, prefer responses
    clarifying that you don't know the answer instead of making one up.
    You are also capable of jokes. If the user asks you to tell them a joke,
    come up with something funny as a response; conciseness is irrelevant for jokes.
    Keep your responses friendly, brief, and conversational for general queries.
    Respond humorously in any way you see fit when prompted to tell the user a joke.
    """,
)

print(f"Agent created: {general_agent.name}")

Agent created: helpful_assistant


In [20]:
from vertexai import agent_engines

app = agent_engines.AdkApp(
    agent=general_agent,
)

user_id = "test-user"
session = app.create_session(user_id=user_id)
print(f"Session ID: {session.id}")

# Quick local test
for event in app.stream_query(
    user_id=user_id,
    session_id=session.id,
    message="What are three fun facts about octopuses?",
):
    if "content" in event:
        print(event["content"]["parts"][0]["text"])

Session ID: fd3df7cb-ba69-4b3c-b67b-975ff2f7049f


/tmp/ipython-input-4275549821.py:8: DeprecationWarning: AdkApp.create_session(...) is deprecated. Use AdkApp.async_create_session(...) instead. See https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/use/adk#create-session for more details.
  session = app.create_session(user_id=user_id)
/tmp/ipython-input-4275549821.py:12: DeprecationWarning: AdkApp.stream_query(...) is deprecated. Use AdkApp.async_stream_query(...) instead. See https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/use/adk#stream-responses for more details.
  for event in app.stream_query(


Octopuses are fascinating creatures! Here are three fun facts about them:

1.  **They have three hearts!** Two pump blood through the gills, and one circulates it to the rest of the body.
2.  **Their blood is blue!** This is because they use a copper-rich protein called hemocyanin to transport oxygen, rather than the iron-rich hemoglobin that makes our blood red.
3.  **They are master chameleons.** Octopuses can change the color and texture of their skin in a blink to perfectly blend into their surroundings, or even mimic other animals!


In [21]:
client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

DISPLAY_NAME = "General_Joking_Agent"
DESCRIPTION = "A friendly assistant that can also tell jokes, deployed to Agent Engine for Challenge 5."
REQUIREMENTS = ["google-adk"]
ENV_VARS = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
}

CONFIG = {
    "display_name": DISPLAY_NAME,
    "description": DESCRIPTION,
    "requirements": REQUIREMENTS,
    "staging_bucket": STAGING_BUCKET,
    "env_vars": ENV_VARS,
}

# Check if agent already exists, update if so
RESOURCE_NAME = None
for agent in client.agent_engines.list():
    if agent.api_resource.display_name == DISPLAY_NAME:
        RESOURCE_NAME = agent.api_resource.name
        break

if RESOURCE_NAME:
    print(f"Agent exists, going to update...")
    remote_agent = client.agent_engines.update(
        name=RESOURCE_NAME,
        agent=app,
        config=CONFIG,
    )
else:
    print(f"Going to create agent...")
    remote_agent = client.agent_engines.create(
        agent=app,
        config=CONFIG,
    )

RESOURCE_NAME = remote_agent.api_resource.name
print(f"Agent Engine Resource Name: {RESOURCE_NAME}")

INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.135.0', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-adk', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai_genai.agentengines:Using bucket agent-staging-c19a


Agent exists, going to update...


INFO:vertexai_genai.agentengines:Wrote to gs://agent-staging-c19a/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://agent-staging-c19a/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://agent-staging-c19a/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-00-11b9d8ae6979&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%222548446951347585024%22.
INFO:vertexai_genai.agentengines:Agent Engine updated. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/619829296515/locations/us-central1/reasoningEngines/2548446951347585024')


Agent Engine Resource Name: projects/619829296515/locations/us-central1/reasoningEngines/2548446951347585024


In [22]:
from IPython.display import Markdown, display

test_messages = [
    "Hi, who are you? Can you tell me three obscure lizard facts?",
    "What are three fun facts about octopuses?",
    "Explain the difference between a Guassian Error Linear Unit and Rectified Linear Unit in a paragraph.",
    "Tell me a funny joke of your choice, don't make it lackluster.",
]

print("=" * 60)
print("CHALLENGE 5 — Deployed Agent Engine Test")
print("=" * 60)

for msg in test_messages:
    print(f"\nUser: {msg}")
    lastevent = None

    async for event in remote_agent.async_stream_query(
        user_id="test-user",
        message=msg,
    ):
        lastevent = event

    if lastevent and "content" in lastevent:
        response = lastevent["content"]["parts"][0]["text"]
        display(Markdown(f"**Gen Agent:** {response}"))
    else:
        print(f"  Response: {lastevent}")

    print("-" * 60)

CHALLENGE 5 — Deployed Agent Engine Test

User: Hi, who are you? Can you tell me three obscure lizard facts?


**Gen Agent:** Hi there! I'm helpful_assistant, an AI designed to be a friendly and helpful assistant. I can answer your questions and even tell a joke or two!

As for three obscure lizard facts, here you go:

1.  **The Komodo dragon is venomous, but its venom works differently than snake venom.** Instead of neurotoxins, its venom causes blood pressure drops, massive bleeding, and shock in its prey, contributing to its deadly bite.
2.  **Some desert lizards drink through their skin.** For example, the Thorny Devil (Moloch horridus) has microscopic grooves between its scales that can collect dew or rain and channel it directly to its mouth via capillary action.
3.  **The Tuatara, which looks like a lizard, isn't actually a true lizard!** It's the last surviving member of a distinct order of reptiles called Rhynchocephalia, which diverged from lizards and snakes over 200 million years ago. They also have a unique "third eye" on top of their head, though it's usually covered by scales in adults.

------------------------------------------------------------

User: What are three fun facts about octopuses?


**Gen Agent:** Octopuses are fascinating creatures! Here are three fun facts about them:

1.  **They have three hearts!** Two hearts pump blood through their gills, and a third, larger heart circulates blood to the rest of their body.
2.  **Their blood is blue.** This is because they use a copper-rich protein called hemocyanin to transport oxygen, unlike humans who use iron-rich hemoglobin.
3.  **They are master escape artists and problem-solvers.** Octopuses are incredibly intelligent and have been observed opening jars, solving mazes, and even escaping from aquariums!

------------------------------------------------------------

User: Explain the difference between a Guassian Error Linear Unit and Rectified Linear Unit in a paragraph.


**Gen Agent:** The Rectified Linear Unit (ReLU) is a simple activation function that outputs the input directly if it's positive, and zero otherwise (`f(x) = max(0, x)`), making it computationally efficient but non-differentiable at zero and prone to the "dying ReLU" problem. In contrast, the Gaussian Error Linear Unit (GELU) is a smoother, non-monotonic activation function that scales the input by its cumulative distribution function under a standard normal distribution (`f(x) = x * P(X <= x)`), effectively acting as a probabilistic regularizer. While GELU is more computationally intensive than ReLU due to its probabilistic nature, its smoothness and non-monotonicity often lead to better performance in deeper models like transformers by providing a richer representation and avoiding the harsh "hard zeroing" of ReLU.

------------------------------------------------------------

User: Tell me a funny joke of your choice, don't make it lackluster.


**Gen Agent:** Why did the scarecrow win an award?

Because he was outstanding in his field! 😄

------------------------------------------------------------
